In [11]:
import os
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# CONFIGURATION

In [12]:
project_dir = r"C:\cache\Youtube-ETL_Project"
cleaned_dir = os.path.join(project_dir, "data", "cleaned")
latest_cleaned_path = os.path.join(cleaned_dir, "latest_cleaned_path.txt")

if os.path.exists(latest_cleaned_path):
    with open(latest_cleaned_path, "r", encoding="utf-8") as f:
        cleaned_path = f.read().strip()
else:
    cleaned_files = [
        os.path.join(cleaned_dir, file_name)
        for file_name in os.listdir(cleaned_dir)
        if file_name.startswith("cleaned_youtube_") and file_name.endswith(".csv")
    ]
    if not cleaned_files:
        raise FileNotFoundError(f"No cleaned CSV files found in: {cleaned_dir}")
    cleaned_path = max(cleaned_files, key=os.path.getmtime)

table_name = "youtube_videos"
if_exists_mode = "replace"

print(f"Cleaned CSV: {cleaned_path}")
print(f"Load mode: {if_exists_mode} -> table `{table_name}`")

Cleaned CSV: C:\cache\Youtube-ETL_Project\data\cleaned\cleaned_youtube_Jul_2026.csv
Load mode: replace -> table `youtube_videos`


# DATABASE CONNECTION

In [13]:
load_dotenv()

mysql_host = os.getenv("MYSQL_HOST")
mysql_user = os.getenv("MYSQL_USER")
mysql_password = os.getenv("MYSQL_PASSWORD")
mysql_database = os.getenv("MYSQL_DATABASE")
mysql_port = int(os.getenv("MYSQL_PORT", "3306"))

required_vars = {
    "MYSQL_HOST": mysql_host,
    "MYSQL_USER": mysql_user,
    "MYSQL_PASSWORD": mysql_password,
    "MYSQL_DATABASE": mysql_database,
}

missing_vars = [name for name, value in required_vars.items() if not value]
if missing_vars:
    raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

encoded_password = quote_plus(mysql_password)
server_url = f"mysql+pymysql://{mysql_user}:{encoded_password}@{mysql_host}:{mysql_port}/?charset=utf8mb4"
database_url = f"mysql+pymysql://{mysql_user}:{encoded_password}@{mysql_host}:{mysql_port}/{mysql_database}?charset=utf8mb4"

server_engine = create_engine(server_url)
with server_engine.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS `{mysql_database}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = create_engine(database_url)
print(f"Connected to MySQL database: {mysql_database}")

OperationalError: (pymysql.err.OperationalError) (1045, "Access denied for user 'root'@'localhost' (using password: YES)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

# READ CLEANED CSV

In [ ]:
if not os.path.exists(cleaned_path):
    raise FileNotFoundError(f"Cleaned CSV not found: {cleaned_path}")

df = pd.read_csv(cleaned_path)

if "video_id" not in df.columns:
    raise ValueError("Column `video_id` is required before loading to MySQL")

print(f"Loaded cleaned CSV shape: {df.shape}")
display(df.head())

# LOAD TO MYSQL

In [ ]:
df.to_sql(
    name=table_name,
    con=engine,
    if_exists=if_exists_mode,
    index=False,
    chunksize=1000,
)

with engine.begin() as conn:
    row_count = conn.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar_one()

print(f"Loaded {row_count} rows into `{mysql_database}`.`{table_name}`")